In [3]:
import json
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore') # 忽略由于样本量小可能出现的警告

# ==========================================
# 1. 读取数据
# ==========================================
target_base_dir = Path("/NAS/shith/uplift/results_final")
target_model_name = "TARNET" # 你指定的用来做 T 检验的核心模型

metrics_to_extract = {
    "Test_Target_Y_AUUC": "AUUC",
    "Test_Target_Y_AUQC": "AUQC",
    "Test_Target_Y_Lift@10": "Lift@10",
    "Test_Target_Y_Lift@30": "Lift@30"
}

data_list = []

print("正在读取数据并计算...")
for model_dir in target_base_dir.iterdir():
    if not model_dir.is_dir():
        continue
    model_name = model_dir.name
    
    for seed_dir in model_dir.iterdir():
        if not seed_dir.is_dir():
            continue
        seed_name = seed_dir.name
        json_file = seed_dir / "final_metrics.json"
        
        if json_file.exists():
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    metrics = json.load(f)
                
                row = {'Model': model_name, 'Seed': seed_name}
                for full_metric, short_name in metrics_to_extract.items():
                    row[short_name] = metrics.get(full_metric, np.nan)
                data_list.append(row)
            except Exception as e:
                pass

df = pd.DataFrame(data_list)

if df.empty:
    raise ValueError("没有读取到任何数据，请检查目录路径是否正确！")

# ==========================================
# 2. 生成均值与方差表
# ==========================================
# 按照 Model 分组，计算均值、方差和标准差
agg_funcs = ['mean', 'var', 'std']
stats_df = df.groupby('Model').agg({metric: agg_funcs for metric in metrics_to_extract.values()})

# 制作一张格式化好的展示表 (格式: 均值 ± 标准差 (方差: xx))
formatted_table = pd.DataFrame(index=stats_df.index)

for metric in metrics_to_extract.values():
    means = stats_df[(metric, 'mean')]
    stds = stats_df[(metric, 'std')]
    vars_ = stats_df[(metric, 'var')]
    
    # 格式化字符串，保留 6 位小数
    formatted_table[metric] = [
        f"{m:.6f} ± {s:.6f} (var: {v:.2e})" if pd.notnull(m) else "N/A" 
        for m, s, v in zip(means, stds, vars_)
    ]

print("\n" + "="*60)
print("表 1: 各模型表现 (均值 ± 标准差 (方差))")
print("="*60)
display(formatted_table) # 在 Jupyter 中渲染漂亮表格

# ==========================================
# 3. T-Test 显著性检验 (TARNET vs Best Baseline)
# ==========================================
if target_model_name not in df['Model'].values:
    print(f"\n⚠️ 警告: 你的结果目录中没有找到名为 '{target_model_name}' 的模型。")
    print(f"请确保 {target_base_dir}/{target_model_name} 存在且包含有效的 json 文件，否则无法进行 T 检验。")
else:
    ttest_results = []
    
    for metric in metrics_to_extract.values():
        # 排除目标模型，寻找当前的 Best Baseline (均值最高)
        baselines_df = df[df['Model'] != target_model_name]
        if baselines_df.empty:
            continue
            
        best_base_model = baselines_df.groupby('Model')[metric].mean().idxmax()
        
        # 获取两组的真实数据点数组
        target_vals = df[df['Model'] == target_model_name][metric].dropna().values
        best_vals = df[df['Model'] == best_base_model][metric].dropna().values
        
        # 执行独立样本 T 检验 (Welch's t-test, 假设方差不齐)
        t_stat, p_val = stats.ttest_ind(target_vals, best_vals, equal_var=False)
        
        # 判断显著性星号
        stars = ""
        if pd.notnull(p_val):
            if p_val < 0.01: stars = "***"
            elif p_val < 0.05: stars = "**"
            elif p_val < 0.1: stars = "*"
            
        ttest_results.append({
            "指标 (Metric)": metric,
            "Target 模型": target_model_name,
            "Target 均值": target_vals.mean() if len(target_vals)>0 else np.nan,
            "Best Baseline": best_base_model,
            "Best 均值": best_vals.mean() if len(best_vals)>0 else np.nan,
            "t-statistic": t_stat,
            "p-value": p_val,
            "显著性": stars
        })

    ttest_df = pd.DataFrame(ttest_results)
    
    print("\n" + "="*60)
    print(f"表 2: {target_model_name} 相比 Best Baseline 的显著性检验 (T-Test)")
    print("="*60)
    display(ttest_df)
    
    print("\n💡 显著性说明: *** (p < 0.01), ** (p < 0.05), * (p < 0.1)")

正在读取数据并计算...

表 1: 各模型表现 (均值 ± 标准差 (方差))


,AUUC,AUQC,Lift@10,Lift@30
Model,,,,
CFRNet,0.883210 ± 0.017031 (var: 2.90e-04),0.886480 ± 0.016065 (var: 2.58e-04),0.008477 ± 0.000457 (var: 2.09e-07),0.003449 ± 0.000060 (var: 3.60e-09)
DragonNet,0.889022 ± 0.002280 (var: 5.20e-06),0.892164 ± 0.001049 (var: 1.10e-06),0.008621 ± 0.000106 (var: 1.13e-08),0.003463 ± 0.000015 (var: 2.22e-10)
ECUP,0.893092 ± 0.001005 (var: 1.01e-06),0.897002 ± 0.000935 (var: 8.74e-07),0.008122 ± 0.000276 (var: 7.61e-08),0.003473 ± 0.000026 (var: 6.66e-10)
EFIN,0.908508 ± 0.001322 (var: 1.75e-06),0.911140 ± 0.001139 (var: 1.30e-06),0.008293 ± 0.000167 (var: 2.80e-08),0.003546 ± 0.000008 (var: 6.50e-11)
EUEN_Academic,0.872697 ± nan (var: nan),0.875110 ± nan (var: nan),0.008361 ± nan (var: nan),0.003476 ± nan (var: nan)
MOTTO,0.900017 ± 0.002656 (var: 7.05e-06),0.903461 ± 0.001935 (var: 3.74e-06),0.008314 ± 0.000206 (var: 4.22e-08),0.003459 ± 0.000019 (var: 3.47e-10)
MTMT,0.864288 ± 0.014963 (var: 2.24e-04),0.866987 ± 0.015388 (var: 2.37e-04),0.007798 ± 0.000947 (var: 8.98e-07),0.003303 ± 0.000125 (var: 1.56e-08)
S_Learner,0.895117 ± 0.012143 (var: 1.47e-04),0.897464 ± 0.012778 (var: 1.63e-04),0.008013 ± 0.000134 (var: 1.80e-08),0.003469 ± 0.000011 (var: 1.18e-10)
TARNET,0.913083 ± 0.001957 (var: 3.83e-06),0.916075 ± 0.001743 (var: 3.04e-06),0.008504 ± 0.000088 (var: 7.70e-09),0.003560 ± 0.000047 (var: 2.21e-09)



表 2: TARNET 相比 Best Baseline 的显著性检验 (T-Test)


,指标 (Metric),Target 模型,Target 均值,Best Baseline,Best 均值,t-statistic,p-value,显著性
0,AUUC,TARNET,0.913083,EFIN,0.908508,3.686828,0.014242,**
1,AUQC,TARNET,0.916075,EFIN,0.911140,4.520437,0.006369,***
2,Lift@10,TARNET,0.008504,DragonNet,0.008621,-1.558586,0.196106,
3,Lift@30,TARNET,0.003560,EFIN,0.003546,0.570537,0.605575,



💡 显著性说明: *** (p < 0.01), ** (p < 0.05), * (p < 0.1)
